## Explore Dataset

In [ ]:
import importlib
import geopandas as gpd
import dotenv
import os
import pandas as pd
import shapely
from google.cloud import storage
import matplotlib.pyplot as plt
import numpy as np
import addCSOs
import optimization
# importlib.reload(optimization)

In [ ]:
dotenv.load_dotenv()

## Obtaining data

Keeping all the data in the EPSG 3857 coordinate referencing system (CRS) to work better with distances

### CSO Data

In [ ]:
geo_JSON_url = "https://services-eu1.arcgis.com/MSNNjkZ51iVh8yBj/arcgis/rest/services/Northumbrian_Water_Storm_Overflow_Activity_2_view/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
cso: gpd.GeoDataFrame = gpd.read_file(geo_JSON_url)
cso_all = cso.to_crs(epsg=3857)

In [ ]:
INPUT_PATH = os.getenv("CSO_DATA_LOC")
if INPUT_PATH is None:
    raise ValueError("Input Path not set")
bucketName = INPUT_PATH.split("/")[2]
path_in_bucket = '/'.join(INPUT_PATH.split("/")[3:])
storage_client = storage.Client()
bucket = storage_client.get_bucket(bucketName)
geojson_filenames = ["gs://"+bucketName+'/' +
                     fname.name for fname in bucket.list_blobs(prefix=path_in_bucket) if fname.name.endswith('.geojson')]
cso = gpd.read_file(geojson_filenames[-1])
cso_all = cso.to_crs(epsg=3857)

### River data

In [ ]:
RIVERS_PATH = os.getenv("RIVERS_PATH")
rivers = gpd.read_file(RIVERS_PATH)
rivers = rivers.to_crs(epsg=3857)
rivers_all = rivers.copy()

### Clip to cso bounds

In [ ]:
rivers = rivers_all.clip(cso_all.total_bounds).explode()
cso = cso_all

### Clip to Catchment Data

In [ ]:
cmnt = gpd.read_file(
    "https://environment.data.gov.uk/catchment-planning/OperationalCatchment/3052.geojson"
)
cmnt.to_crs(cso_all.crs, inplace=True)

In [ ]:
rivers = gpd.clip(rivers_all, cmnt)
# Remove all rivers that aren't LineStrings after Clipping
rivers = rivers[rivers.geometry.apply(
    lambda x: isinstance(x, shapely.LineString))]
cso = gpd.clip(cso_all, cmnt)

### Pre Explore

In [ ]:
m = cmnt.explore(highlight=False,
                 popup=False,
                 tooltip=False,
                 color="grey")
rivers.explore(m=m)
cso.explore(m=m, marker_kwds={"radius": 5}, color="red")

## Use addCSOs.py

In [ ]:
RIVER_RESOLUTION: float = 50
MAX_OUTFALL_DIST_FROM_RIVER: float = 700
MAX_RISK_DISTANCE: float = 10_000

In [ ]:
river_points, cso_with_locations = addCSOs.calculate_risk_factors_cso(
    cso,
    rivers,
    cso_df_id_col="id",
    river_resolution=RIVER_RESOLUTION,
    max_outfall_dist_from_river=MAX_OUTFALL_DIST_FROM_RIVER,
    max_risk_distance=MAX_RISK_DISTANCE,
    include_upstream=False,
    river_directed=True,
)

## Add Risk

In [ ]:
def linear_dist_decrease(d: float) -> float:
    return (MAX_RISK_DISTANCE-d)/MAX_RISK_DISTANCE

In [ ]:
def add_simple_risk(gdf: gpd.GeoDataFrame) -> gpd.GeoSeries:
    return gdf["csoInfo"].apply(lambda x: sum([linear_dist_decrease(dist) for _, dist in x]))

In [ ]:
def add_frequency_risk(gdf: gpd.GeoDataFrame) -> gpd.GeoSeries:
    EDM_PATH = os.getenv("CSO_DATA_OUTPUT")
    edm = pd.read_pickle(EDM_PATH)
    freq = edm.groupby("Monitor_ID").count()["Event_Start"]
    return gdf["csoInfo"].apply(lambda x: sum([freq[csoid]*linear_dist_decrease(dist) for csoid, dist in x if csoid in freq.index]))

In [ ]:
def add_duration_risk(gdf: gpd.GeoDataFrame) -> gpd.GeoSeries:
    EDM_PATH = os.getenv("CSO_DATA_OUTPUT")
    edm = pd.read_pickle(EDM_PATH)
    dur = edm["Event_End"] - edm["Event_Start"]
    duration = dur.groupby("Monitor_ID").sum()
    return gdf["csoInfo"].apply(lambda x: sum([duration[csoid].total_seconds()/60/60*linear_dist_decrease(dist) for csoid, dist in x if csoid in duration.index]))

In [ ]:
river_points["Risk"] = add_duration_risk(river_points)

In [ ]:
river_points_risk = river_points[river_points["Risk"] > 0]

## Explore

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

rgmap = LinearSegmentedColormap.from_list('rgmap', ['g', 'r'], N=100)

In [ ]:
m = cmnt.explore(
    highlight=False,
    tooltip=False,
    popup=False,
    color="grey",
)
rivers.explore(
    m=m,
    color="blue",
)
river_points_risk.explore(
    m=m,
    column="Risk",
    cmap='viridis',
    # vmax=150,
    marker_kwds={"radius": 5},
)
# cso_with_locations.explore(
#     m=m,
#     color="red",
#     marker_kwds={"radius": 8}
# )
cso_with_locations.set_geometry("outfallLocation").explore(
    m=m,
    color="pink",
    marker_kwds={"radius": 8}
)

## Optimize Placement of Sensors

In [ ]:
N_SENSORS = 5
sensors = optimization.greedy_optimization(
    N_SENSORS, river_points_risk, add_duration_risk)

In [ ]:
m = sensors.explore(
    color="red",
    marker_kwds={"radius": 6},
)
river_points_risk.explore(
    m=m,
    column="Risk")

## Use WQ Data

In [ ]:
wqData = gpd.read_file(os.getenv("WQ_DATA_PATH"))

In [ ]:
wqData = gpd.clip(wqData, cmnt)

In [ ]:
m = cmnt.explore(highlight=False,
                 popup=False,
                 tooltip=False,
                 color="grey")
rivers.explore(m=m)
wqData.explore(m=m, marker_kwds={"radius": 5}, color="red")
cso.explore(m=m, marker_kwds={"radius": 5}, color="dark green")